# K16 Pisa v15 progressive GPU islands

Witness-only CUDA companion to GitHub's exact v15 campaign. It triages every still-relevant endpoint, retains the best 40%, then the best 30%, and gives the survivors progressively longer budgets. A miss is not an UNSAT result; any hit is independently verified.

In [ ]:
import os, shutil, subprocess, torch

REPO = "https://github.com/lieoric/k16-pisa-terminal-search.git"
REF = "agent/k16-endpoint-endgame"
ROOT = "/kaggle/working/k16-pisa-terminal-search"
OUTPUT = "/kaggle/working/k16_gpu_v15"

if os.path.exists(ROOT):
    shutil.rmtree(ROOT)
subprocess.run(["git", "clone", "--depth", "1", "--branch", REF, REPO, ROOT], check=True)
os.chdir(ROOT)

gpu_count = torch.cuda.device_count()
if gpu_count < 1:
    raise RuntimeError("Select a Kaggle GPU accelerator")

# Keep total wall time near seven hours whether Kaggle grants one or two GPUs.
budgets = "180,900,3600" if gpu_count >= 2 else "90,450,1800"
commands = []
for worker in range(gpu_count):
    commands.append([
        "python", "scripts/kaggle_v15_gpu_progressive.py",
        "--device", f"cuda:{worker}",
        "--worker-index", str(worker),
        "--worker-count", str(gpu_count),
        "--budgets", budgets,
        "--keep-fractions", "0.4,0.3",
        "--batch-size", "16384",
        "--min-total-b", "16",
        "--seed-offset", "1500000",
        "--output-dir", OUTPUT,
    ])

processes = [subprocess.Popen(command) for command in commands]
codes = [process.wait() for process in processes]
if any(codes):
    raise RuntimeError(f"GPU worker failure: {codes}")
print("K16_V15_GPU_CAMPAIGN_DONE", OUTPUT)
